# Preprocessing step by step 

This notebook presents step-by-step preprocessing steps to get to know the dataset and evaluate its features

In [133]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### 1. Load the dataset 
Load the data and print some of the attributes

In [134]:
df = pd.read_csv('../dataset/training_data.csv')

print(df.head())

                                               title                date  \
0  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
1  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
2  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
3  The President's News Conference in Hanoi, Vietnam  September 10, 2023   
4  The President's News Conference in Hanoi, Vietnam  September 10, 2023   

         president                                                url  \
0  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
1  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
2  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
3  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   
4  Joseph R. Biden  https://www.presidency.ucsb.edu/documents/the-...   

   question_order                                 interview_question  \
0               1  Q. Of the Bid

In [135]:
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3448 entries, 0 to 3447
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   title                  3448 non-null   object 
 1   date                   3448 non-null   object 
 2   president              3448 non-null   object 
 3   url                    3448 non-null   object 
 4   question_order         3448 non-null   int64  
 5   interview_question     3448 non-null   object 
 6   interview_answer       3448 non-null   object 
 7   gpt3.5_summary         3448 non-null   object 
 8   gpt3.5_prediction      3448 non-null   object 
 9   question               3448 non-null   object 
 10  annotator_id           3448 non-null   int64  
 11  annotator1             0 non-null      float64
 12  annotator2             0 non-null      float64
 13  annotator3             0 non-null      float64
 14  inaudible              3448 non-null   bool   
 15  mult

In [136]:
print(df.describe(include="all"))

                                  title               date        president  \
count                              3448               3448             3448   
unique                              175                287                4   
top     The President's News Conference  November 07, 2018  Donald J. Trump   
freq                               1626                117             1325   
mean                                NaN                NaN              NaN   
std                                 NaN                NaN              NaN   
min                                 NaN                NaN              NaN   
25%                                 NaN                NaN              NaN   
50%                                 NaN                NaN              NaN   
75%                                 NaN                NaN              NaN   
max                                 NaN                NaN              NaN   

                                                   

Check for missing values

In [137]:
print(df.isna().sum())

title                       0
date                        0
president                   0
url                         0
question_order              0
interview_question          0
interview_answer            0
gpt3.5_summary              0
gpt3.5_prediction           0
question                    0
annotator_id                0
annotator1               3448
annotator2               3448
annotator3               3448
inaudible                   0
multiple_questions          0
affirmative_questions       0
index                       0
clarity_label               0
evasion_label               0
dtype: int64


### 2. Data cleanup

Start cleaning the dataset by removing unnecessary features

Remove null-fetaures and urls.

In [138]:

for i in [1, 2, 3]:
    annotator = 'annotator' + str(i)
    df = df.drop(annotator, axis=1)
df = df.drop('url', axis = 1)

### 3. Q&A mathing 

Distinguish and separate multiple part questions in to separate rows

In [93]:
import re

Start by defining helper functions to separate questions and answers

In [154]:
import re

def gpt_summary_parser(summary):
    if not isinstance(summary, str):
        return [], []

    # Split questions vs. answers
    parts = re.split(r'The response provides[^:]*:\s*',
                     summary, maxsplit=1, flags=re.IGNORECASE)
    if len(parts) < 2:
        return [], []

    q_block, rest = parts
    # Remove any "Overall ..." section
    a_block = re.split(r'\bOverall\b', rest, maxsplit=1, flags=re.IGNORECASE)[0]

    # 1) Try numbered questions: "1. Question..."
    q_list = [m.strip() for m in re.findall(r'\d+\.\s*(.+)', q_block)]

    # 2) Fallback: "Part 1: Question..."
    if not q_list:
        q_list = [m.strip() for m in re.findall(r'Part\s+\d+:\s*(.+)', q_block, flags=re.IGNORECASE)]

    # 3) Fallback: “The question consists of 1 part: <question>”
    if not q_list:
        m = re.search(r'The question consists of 1 part:?\s*(.+)',
                      q_block, flags=re.IGNORECASE | re.DOTALL)
        if m:
            tail = m.group(1).strip()
            first_line = tail.splitlines()[0].strip()
            if first_line:
                q_list = [first_line]

    # Answers: try numbered list "1. ..."
    # a_list = [m.strip() for m in re.findall(r'\d+\.\s*(.+)', a_block)]

    if re.search(r'\d+\.\s*Q\d', a_block):
        # Treat as bullet answers instead
        a_list = [
            m.strip()
            for m in re.findall(r'^\s*[-•]\s*(.+)', a_block, flags=re.MULTILINE)
        ]
    else:
        # Normal numbered answers
        a_list = [
            m.strip()
            for m in re.findall(r'\d+\.\s*(.+)', a_block)
        ]

    # If there’s exactly one question but no structured answers,
    # treat the whole block as a single answer
    if len(q_list) == 1 and not a_list:
        cleaned = re.sub(r'^\s*[-•]\s*', '', a_block.strip(), flags=re.MULTILINE)
        a_list = [cleaned.strip()]

    if len(a_list) != len(q_list):
        a_list = a_list[:-1]
        
    return q_list, a_list

In [155]:

def extract_single_answer(summary: str) -> str | None:
    """
    For summaries where there's effectively only one question,
    return everything starting from 'The response...' onward.
    """
    if not isinstance(summary, str):
        return None
    
    s_lower = summary.lower()
    
    # Best case: there's a 'The response ...' phrase
    idx = s_lower.find("the response")
    if idx != -1:
        return summary[idx:].strip()
    
    # Fallback: if 'The question consists' exists, take text after it
    idx2 = s_lower.find("the question consists")
    if idx2 != -1:
        parts = summary[idx2:].split("\n", 1)
        if len(parts) == 2:
            return parts[1].strip()
    
    # Last resort: whole summary
    return summary.strip()

In [161]:
def normalize_question(q: str) -> str:
    if not isinstance(q, str):
        return ""
    q = re.sub(r'^Q\d+:\s*', '', q)  # remove "Q1:", "Q2:", etc
    return q.strip()

df['answer'] = None

row = df.iloc[24]
q, a = gpt_summary_parser(row['gpt3.5_summary'])

print(q, a)

for summary, group in df.groupby('gpt3.5_summary'):
    q_list, a_list = gpt_summary_parser(summary)

    unique_questions = group['question']
    if (not q_list or not a_list) and len(unique_questions) == 1:
        single_ans = extract_single_answer(summary)
        df.loc[group.index, 'answer'] = single_ans
        continue

    if not q_list or not a_list:
        continue

    # normalize keys
    mapping = {
        normalize_question(q): a.strip()
        for q, a in zip(q_list, a_list)
    }
    for idx, row in group.iterrows():
        q_text = normalize_question(str(row['question']))
        subans = mapping.get(q_text)
        df.at[idx, 'answer'] = subans

['Does this raise any new concerns about Putin potentially doing more drastic things regarding Ukraine, like nuclear weapons, or potentially against the U.S., like election interference?', 'Does the firing of the general and the rebellion by Prigozhin indicate any potential future actions by Putin?'] ["Regarding concerns about Putin potentially using nuclear weapons, the response states that not only the West but also China and the rest of the world have warned against it. Therefore, there isn't any real prospect of Putin using nuclear weapons.", 'Regarding concerns about Putin potentially interfering in U.S. elections, the response acknowledges that they have already interfered in American elections in the past.']
{'Clarification: What is meant by "steps" in this question?': 'Clarification on "steps": The response mentions taking administrative steps instead of legislative ones to gather data on guns that fall into the hands of criminals and track them more effectively.', 'Request for

Use the helper function to split the summary in to Q/A pairs and add new attributes to the dataset. 

In [160]:

df[["interview_question", "question", "answer", 'evasion_label']].head(40)

# df['answer'].head(10)

,interview_question,question,answer,evasion_label
0,Q. Of the Biden administration. And accused th...,How would you respond to the accusation that t...,The President expresses sincerity about gettin...,Explicit
1,Q. Of the Biden administration. And accused th...,Do you think President Xi is being sincere abo...,China is changing some of the rules of the gam...,General
2,Q. No worries. Do you believe the country's sl...,Do you believe the country's slowdown and gro...,The President acknowledges that China has a di...,Partial/half-answer
3,Q. No worries. Do you believe the country's sl...,Are you worried about the meeting between Pre...,None,Dodging
4,"Q. I can imagine. It is evening, I'd like to r...",Is the President's engagement with Asian coun...,The President mentions the deals and pacts sig...,Explicit
5,"Q. I can imagine. It is evening, I'd like to r...",Is there a danger of a cold war?,The President emphasizes that his approach is ...,Implicit
6,"Q. I can imagine. It is evening, I'd like to r...",When will the President meet Mr. Xi?,The President expresses his hope to meet Mr. X...,Deflection
7,Q. It's Aurelia End for AFP. I had a question ...,How concerned are you about this lack of cons...,- The concept of building economic growth from...,Implicit
8,"Q. Well, let me ask you about—you've spent lot...",Concerns about the lack of communication betw...,The interviewee states that although they have...,Explicit
9,"Q. Well, let me ask you about—you've spent lot...",Inquiry about the reaction of Kyiv regarding ...,The interviewee acknowledges that the issue of...,Explicit


In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3448 entries, 0 to 3447
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   title                  3448 non-null   object
 1   date                   3448 non-null   object
 2   president              3448 non-null   object
 3   question_order         3448 non-null   int64 
 4   interview_question     3448 non-null   object
 5   interview_answer       3448 non-null   object
 6   gpt3.5_summary         3448 non-null   object
 7   gpt3.5_prediction      3448 non-null   object
 8   question               3448 non-null   object
 9   annotator_id           3448 non-null   int64 
 10  inaudible              3448 non-null   bool  
 11  multiple_questions     3448 non-null   bool  
 12  affirmative_questions  3448 non-null   bool  
 13  index                  3448 non-null   int64 
 14  clarity_label          3448 non-null   object
 15  evasion_label        

### 4. Save file 

Save the new dataset to a CSV file

In [131]:
df.to_csv('../dataset/training_data_processed.csv')